# 1) Load CSVs and Summarize Order Demand
This block reads the three source CSV files and prints item quantities needed per order.


In [40]:
import pandas as pd

itemtypes = pd.read_csv("order_itemtypes.csv", header=None)
quantities = pd.read_csv("order_quantities.csv", header=None)
totes = pd.read_csv("orders_totes.csv", header=None)

order_rows = []
for o in range(itemtypes.shape[0]):
    for k in range(itemtypes.shape[1]):
        it = itemtypes.iat[o, k]
        qt = quantities.iat[o, k] if k < quantities.shape[1] else pd.NA
        tt = totes.iat[o, k] if k < totes.shape[1] else pd.NA
        if pd.notna(it) and pd.notna(qt) and pd.notna(tt):
            order_rows.append({"order": o + 1, "item_type": int(it), "quantity": int(qt)})

order_df = pd.DataFrame(order_rows)
if order_df.empty:
    print("No valid order-item rows found in CSV files.")
else:
    order_summary = order_df.groupby(["order", "item_type"], as_index=False)["quantity"].sum()
    for order_id, grp in order_summary.groupby("order"):
        parts = [f"item {int(r.item_type)} x {int(r.quantity)}" for r in grp.itertuples(index=False)]
        print(f"Order {int(order_id)}: " + ", ".join(parts))


Order 1: item 1 x 2, item 3 x 3
Order 2: item 2 x 1, item 3 x 3, item 4 x 1
Order 3: item 5 x 2
Order 4: item 0 x 3, item 5 x 1
Order 5: item 1 x 1, item 2 x 1
Order 6: item 1 x 2
Order 7: item 3 x 2, item 5 x 1
Order 8: item 2 x 1, item 4 x 2
Order 9: item 0 x 3, item 1 x 2, item 5 x 1
Order 10: item 6 x 3
Order 11: item 1 x 1


# 2) Load CSVs and Summarize Tote Contents
This block reads the same CSV files and prints which item types/quantities are in each tote.


In [41]:
import pandas as pd

itemtypes = pd.read_csv("order_itemtypes.csv", header=None)
quantities = pd.read_csv("order_quantities.csv", header=None)
totes = pd.read_csv("orders_totes.csv", header=None)

tote_rows = []
for o in range(itemtypes.shape[0]):
    for k in range(itemtypes.shape[1]):
        it = itemtypes.iat[o, k]
        qt = quantities.iat[o, k] if k < quantities.shape[1] else pd.NA
        tt = totes.iat[o, k] if k < totes.shape[1] else pd.NA
        if pd.notna(it) and pd.notna(qt) and pd.notna(tt):
            tote_rows.append({"tote": int(tt), "item_type": int(it), "quantity": int(qt)})

tote_df = pd.DataFrame(tote_rows)
if tote_df.empty:
    print("No valid tote-item rows found in CSV files.")
else:
    tote_summary = tote_df.groupby(["tote", "item_type"], as_index=False)["quantity"].sum()
    for tote_id, grp in tote_summary.sort_values(["tote", "item_type"]).groupby("tote"):
        parts = [f"item {int(r.item_type)} x {int(r.quantity)}" for r in grp.itertuples(index=False)]
        print(f"Tote {int(tote_id)}: " + ", ".join(parts))


Tote 0: item 1 x 2, item 3 x 3
Tote 1: item 1 x 2, item 6 x 3
Tote 3: item 5 x 1
Tote 4: item 3 x 3
Tote 7: item 5 x 2
Tote 8: item 0 x 3, item 2 x 1, item 4 x 2
Tote 9: item 0 x 3
Tote 10: item 1 x 1, item 5 x 1
Tote 11: item 1 x 1
Tote 12: item 2 x 1
Tote 13: item 1 x 2, item 3 x 2, item 5 x 1
Tote 14: item 2 x 1, item 4 x 1


# 3) MILP Optimization (Order Sequence, Tote Sequence, Item Sequence, and Conveyor Assignment)
This block builds and solves the MILP with recirculation-aware timing, then prints and exports results.


In [42]:
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy.sparse import lil_matrix


def load_units(order_itemtypes_csv, order_quantities_csv, order_totes_csv):
    itemtypes = pd.read_csv(order_itemtypes_csv, header=None)
    quantities = pd.read_csv(order_quantities_csv, header=None)
    totes = pd.read_csv(order_totes_csv, header=None)

    if not (itemtypes.shape[0] == quantities.shape[0] == totes.shape[0]):
        raise ValueError("CSV row counts must match (one row per order)")

    units = []
    for o in range(itemtypes.shape[0]):
        for k in range(itemtypes.shape[1]):
            it = itemtypes.iat[o, k]
            qt = quantities.iat[o, k] if k < quantities.shape[1] else np.nan
            tt = totes.iat[o, k] if k < totes.shape[1] else np.nan
            if pd.notna(it) and pd.notna(qt) and pd.notna(tt):
                q = int(qt)
                if q < 0:
                    raise ValueError(f"Negative quantity at order {o + 1}, column {k + 1}")
                for _ in range(q):
                    units.append({"order": o + 1, "item_type": int(it), "tote": int(tt)})

    units_df = pd.DataFrame(units)
    if units_df.empty:
        raise ValueError("No units were created, check CSV contents")

    return units_df


def solve_milp(
    units_df,
    g=5,
    precedence_pairs=None,
    minimize="sum_completion",
    min_gap=0,
    time_limit=2400,
    mip_rel_gap=0.02,
    retry_time_limits=(2400, None),
):
    """
    Decision variables include:
      1) sequence of orders,
      2) sequence of items within totes,
      3) sequence of tote release,
      4) order-to-conveyor assignment.

    Recirculation timing:
      pick_time = release_time + base_offset(conveyor) + 3*g*recirc_loops (for conveyors 1..3)
      conveyor 0 has no recirculation return path in this model.
    """

    N = len(units_df)
    orders = sorted(units_df["order"].unique().tolist())
    On = len(orders)
    order_to_i = {o: i for i, o in enumerate(orders)}

    tote_vals = sorted(units_df["tote"].unique().tolist())
    Tn = len(tote_vals)
    tote_to_i = {t: i for i, t in enumerate(tote_vals)}

    units_df = units_df.copy().reset_index(drop=True)
    units_df["order_i"] = units_df["order"].map(order_to_i)
    units_df["tote_i"] = units_df["tote"].map(tote_to_i)

    L = np.zeros(Tn, dtype=int)
    for ti in range(Tn):
        L[ti] = int((units_df["tote_i"] == ti).sum())

    if precedence_pairs is None:
        precedence_pairs = []
    if min_gap < 0:
        raise ValueError("min_gap must be >= 0")

    offset = 0

    # x[u,s] assignment of unit u to release slot s
    x_offset = offset
    x_size = N * N
    offset += x_size

    # w[t,s] slot s belongs to tote t block
    w_offset = offset
    w_size = Tn * N
    offset += w_size

    # y[t,k] tote t starts at slot k
    y_starts = []
    y_sizes = []
    for ti in range(Tn):
        y_starts.append(offset)
        sz = N - L[ti] + 1
        y_sizes.append(sz)
        offset += sz
    y_total = sum(y_sizes)

    # order completion and start times
    C_offset = offset
    C_size = On
    offset += C_size

    S_offset = offset
    S_size = On
    offset += S_size

    Cmax_offset = None
    if minimize == "makespan":
        Cmax_offset = offset
        offset += 1

    # p[u] release slot index of unit u
    p_offset = offset
    p_size = N
    offset += p_size

    # k[u] recirculation loop count of unit u
    k_offset = offset
    k_size = N
    offset += k_size

    # a[o,c] order o assigned to conveyor c
    a_offset = offset
    a_size = On * 4
    offset += a_size

    same_order_pairs = []
    if min_gap > 0:
        for o in orders:
            idxs = units_df.index[units_df["order"] == o].tolist()
            for i in range(len(idxs)):
                for j in range(i + 1, len(idxs)):
                    same_order_pairs.append((idxs[i], idxs[j]))

    z_gap_offset = offset
    z_gap_size = len(same_order_pairs)
    offset += z_gap_size

    # r[i,j] pairwise order sequencing binary for i<j
    order_pairs = [(i, j) for i in range(On) for j in range(i + 1, On)]
    r_offset = offset
    r_size = len(order_pairs)
    offset += r_size

    num_vars = offset

    def idx_x(u, s):
        return x_offset + u * N + s

    def idx_w(ti, s):
        return w_offset + ti * N + s

    def idx_y(ti, k):
        return y_starts[ti] + k

    def idx_C(oi):
        return C_offset + oi

    def idx_S(oi):
        return S_offset + oi

    def idx_Cmax():
        return Cmax_offset

    def idx_p(u):
        return p_offset + u

    def idx_k(u):
        return k_offset + u

    def idx_a(oi, c):
        return a_offset + oi * 4 + c

    def idx_z(pair_i):
        return z_gap_offset + pair_i

    def idx_r(pair_i):
        return r_offset + pair_i

    c = np.zeros(num_vars)
    if minimize == "sum_completion":
        c[C_offset:C_offset + C_size] = 1.0
    elif minimize == "makespan":
        c[idx_Cmax()] = 1.0
    else:
        raise ValueError("minimize must be sum_completion or makespan")

    integrality = np.zeros(num_vars, dtype=int)
    integrality[x_offset:x_offset + x_size] = 1
    integrality[w_offset:w_offset + w_size] = 1
    integrality[y_starts[0]:y_starts[0] + y_total] = 1
    integrality[k_offset:k_offset + k_size] = 1
    integrality[a_offset:a_offset + a_size] = 1
    if z_gap_size > 0:
        integrality[z_gap_offset:z_gap_offset + z_gap_size] = 1
    if r_size > 0:
        integrality[r_offset:r_offset + r_size] = 1

    lb = np.zeros(num_vars)
    ub = np.ones(num_vars)

    ub[p_offset:p_offset + p_size] = N - 1
    ub[k_offset:k_offset + k_size] = N

    # Conservative time bounds (includes possible recirculation loops)
    max_time = (N - 1) * g + 4 * g + 3 * g * N
    ub[C_offset:C_offset + C_size] = max_time
    ub[S_offset:S_offset + S_size] = max_time
    if minimize == "makespan":
        ub[idx_Cmax()] = max_time

    bounds = Bounds(lb, ub)

    eq_rows = []
    eq_rhs = []

    # each unit exactly one slot
    for u in range(N):
        eq_rows.append({idx_x(u, s): 1.0 for s in range(N)})
        eq_rhs.append(1.0)

    # each slot exactly one unit
    for s in range(N):
        eq_rows.append({idx_x(u, s): 1.0 for u in range(N)})
        eq_rhs.append(1.0)

    # each tote one start slot
    for ti in range(Tn):
        eq_rows.append({idx_y(ti, k): 1.0 for k in range(y_sizes[ti])})
        eq_rhs.append(1.0)

    # tote coverage constraints
    for ti in range(Tn):
        Lt = L[ti]
        for s in range(N):
            row = {idx_w(ti, s): 1.0}
            k_min = max(0, s - (Lt - 1))
            k_max = min(y_sizes[ti] - 1, s)
            for k in range(k_min, k_max + 1):
                row[idx_y(ti, k)] = row.get(idx_y(ti, k), 0.0) - 1.0
            eq_rows.append(row)
            eq_rhs.append(0.0)

    # slot belongs to one tote block
    for s in range(N):
        eq_rows.append({idx_w(ti, s): 1.0 for ti in range(Tn)})
        eq_rhs.append(1.0)

    # define p[u] from x
    for u in range(N):
        row = {idx_p(u): 1.0}
        for s in range(N):
            row[idx_x(u, s)] = row.get(idx_x(u, s), 0.0) - float(s)
        eq_rows.append(row)
        eq_rhs.append(0.0)

    # each order assigned to exactly one conveyor
    for oi in range(On):
        eq_rows.append({idx_a(oi, c): 1.0 for c in range(4)})
        eq_rhs.append(1.0)

    Aeq = lil_matrix((len(eq_rhs), num_vars), dtype=float)
    beq = np.array(eq_rhs, dtype=float)
    for r, row in enumerate(eq_rows):
        for j, v in row.items():
            Aeq[r, j] = v
    eq_con = LinearConstraint(Aeq.tocsc(), beq, beq)

    ub_rows = []
    ub_rhs = []

    M_time = max_time + 5 * g

    # unit can only appear in its tote's active slots
    for u in range(N):
        ti = int(units_df.loc[u, "tote_i"])
        for s in range(N):
            ub_rows.append({idx_x(u, s): 1.0, idx_w(ti, s): -1.0})
            ub_rhs.append(0.0)

    # conveyor 0 has no recirculation loops
    # k[u] <= N * (1 - a[o,0])
    for u in range(N):
        oi = int(units_df.loc[u, "order_i"])
        ub_rows.append({idx_k(u): 1.0, idx_a(oi, 0): float(N)})
        ub_rhs.append(float(N))

    # Timing constraints linking pick times to assignment a[o,c]
    for u in range(N):
        oi = int(units_df.loc[u, "order_i"])

        for cidx in range(4):
            base_offset = (cidx + 1) * g
            loop_period = 0 if cidx == 0 else 3 * g

            # C[o] >= g*p + base + loop*k - M*(1-a[o,c])
            # -> g*p + loop*k - C + M*a[o,c] <= M - base
            row_c = {idx_p(u): float(g), idx_C(oi): -1.0, idx_a(oi, cidx): float(M_time)}
            if loop_period > 0:
                row_c[idx_k(u)] = float(loop_period)
            ub_rows.append(row_c)
            ub_rhs.append(float(M_time - base_offset))

            # S[o] <= g*p + base + loop*k + M*(1-a[o,c])
            # -> S - g*p - loop*k + M*a[o,c] <= M + base
            row_s = {idx_S(oi): 1.0, idx_p(u): -float(g), idx_a(oi, cidx): float(M_time)}
            if loop_period > 0:
                row_s[idx_k(u)] = -float(loop_period)
            ub_rows.append(row_s)
            ub_rhs.append(float(M_time + base_offset))

    # start <= completion for each order
    for oi in range(On):
        ub_rows.append({idx_S(oi): 1.0, idx_C(oi): -1.0})
        ub_rhs.append(0.0)

    # one order completes before another starts (sequence decided by MILP)
    for pair_i, (i, j) in enumerate(order_pairs):
        r_ij = idx_r(pair_i)

        # if r_ij=1, i before j: C[i] <= S[j]
        ub_rows.append({idx_C(i): 1.0, idx_S(j): -1.0, r_ij: float(M_time)})
        ub_rhs.append(float(M_time))

        # if r_ij=0, j before i: C[j] <= S[i]
        ub_rows.append({idx_C(j): 1.0, idx_S(i): -1.0, r_ij: -float(M_time)})
        ub_rhs.append(0.0)

    # optional fixed precedence pairs
    for (o1, o2) in precedence_pairs:
        if o1 not in order_to_i or o2 not in order_to_i:
            raise ValueError(f"precedence pair includes unknown order, {(o1, o2)}")
        i1 = order_to_i[o1]
        i2 = order_to_i[o2]
        ub_rows.append({idx_C(i1): 1.0, idx_S(i2): -1.0})
        ub_rhs.append(0.0)

    if minimize == "makespan":
        for oi in range(On):
            ub_rows.append({idx_C(oi): 1.0, idx_Cmax(): -1.0})
            ub_rhs.append(0.0)

    # optional same-order spacing
    if min_gap > 0:
        for pair_i, (u, v) in enumerate(same_order_pairs):
            z = idx_z(pair_i)
            ub_rows.append({idx_p(v): 1.0, idx_p(u): -1.0, z: float(N)})
            ub_rhs.append(float(N - min_gap))
            ub_rows.append({idx_p(u): 1.0, idx_p(v): -1.0, z: -float(N)})
            ub_rhs.append(float(-min_gap))

    Aub = lil_matrix((len(ub_rhs), num_vars), dtype=float)
    bub = np.array(ub_rhs, dtype=float)
    for r, row in enumerate(ub_rows):
        for j, v in row.items():
            Aub[r, j] = v
    ub_con = LinearConstraint(Aub.tocsc(), -np.inf * np.ones_like(bub), bub)

    def run_solver(limit, gap):
        options = {"disp": True, "presolve": True}
        if limit is not None:
            options["time_limit"] = float(limit)
        if gap is not None:
            options["mip_rel_gap"] = float(gap)
        return milp(c=c, integrality=integrality, bounds=bounds, constraints=[eq_con, ub_con], options=options)

    res = run_solver(time_limit, mip_rel_gap)

    # Retry if first solve did not produce an incumbent.
    if res.x is None:
        for tl in retry_time_limits:
            retry_gap = 0.08 if tl is not None else None
            res = run_solver(tl, retry_gap)
            if res.x is not None:
                break

    if res.status not in (0, 1) or res.x is None:
        raise RuntimeError(f"No feasible solution found, status {res.status}, message {res.message}")

    v = res.x

    # Decode assignment
    order_to_conveyor = {}
    for oi, o in enumerate(orders):
        a_vals = [v[idx_a(oi, c)] for c in range(4)]
        order_to_conveyor[o] = int(np.argmax(a_vals))

    x_sol = v[x_offset:x_offset + x_size].reshape((N, N))
    slot_to_unit = np.argmax(x_sol, axis=0)
    p_sol = v[p_offset:p_offset + p_size]
    k_sol = v[k_offset:k_offset + k_size]

    belt = []
    for s in range(N):
        u = int(slot_to_unit[s])
        order_id = int(units_df.loc[u, "order"])
        conv = order_to_conveyor[order_id]

        release_time = float(round(float(p_sol[u])) * g)
        base_offset = (conv + 1) * g
        loop_period = 0 if conv == 0 else 3 * g
        loops = int(round(float(k_sol[u])))
        pick_time = release_time + base_offset + loop_period * loops

        belt.append(
            {
                "slot": s + 1,
                "release_time_seconds": release_time,
                "pick_time_seconds": pick_time,
                "unit": u,
                "order": order_id,
                "item_type": int(units_df.loc[u, "item_type"]),
                "tote": int(units_df.loc[u, "tote"]),
                "conveyor": conv,
                "recirc_loops": loops,
            }
        )

    belt_df = pd.DataFrame(belt).sort_values("slot").reset_index(drop=True)

    tote_blocks = (
        belt_df.groupby("tote")
        .agg(start_slot=("slot", "min"), end_slot=("slot", "max"), units=("slot", "count"))
        .sort_values("start_slot")
    )

    order_start_seconds = belt_df.groupby("order")["pick_time_seconds"].min().sort_index()
    order_completion_seconds = belt_df.groupby("order")["pick_time_seconds"].max().sort_index()
    order_start_slots = belt_df.groupby("order")["slot"].min().sort_index()
    order_completion_slots = belt_df.groupby("order")["slot"].max().sort_index()
    order_sequence = order_start_seconds.sort_values().index.tolist()

    tote_sequence = tote_blocks.index.tolist()
    tote_item_sequence = (
        belt_df[["tote", "slot", "pick_time_seconds", "item_type", "order", "unit", "conveyor", "recirc_loops"]]
        .sort_values(["tote", "slot"])
        .reset_index(drop=True)
    )

    return {
        "objective_value_seconds": float(order_completion_seconds.sum()),
        "solver_status": int(res.status),
        "solver_message": str(res.message),
        "order_to_conveyor": order_to_conveyor,
        "belt_sequence": belt_df,
        "tote_blocks": tote_blocks,
        "tote_sequence": tote_sequence,
        "tote_item_sequence": tote_item_sequence,
        "order_sequence": order_sequence,
        "order_start_slots": order_start_slots,
        "order_start_seconds": order_start_seconds,
        "order_completion_slots": order_completion_slots,
        "order_completion_seconds": order_completion_seconds,
    }


if __name__ == "__main__":
    units_df = load_units("order_itemtypes.csv", "order_quantities.csv", "orders_totes.csv")

    sol = solve_milp(
        units_df,
        g=5,
        precedence_pairs=[],
        minimize="sum_completion",
        min_gap=0,
        time_limit=2400,
        mip_rel_gap=0.02,
        retry_time_limits=(2400, None),
    )

    print("\nSolver status")
    print(sol["solver_status"], sol["solver_message"])

    print("\nOrder -> conveyor")
    print(sol["order_to_conveyor"])

    print("\nObjective (sum completion time in seconds, recirculation-aware)")
    print(sol["objective_value_seconds"])

    print("\nChosen order sequence")
    print(sol["order_sequence"])

    print("\nTote sequence")
    print(sol["tote_sequence"])

    print("\nOrder completion seconds")
    print(sol["order_completion_seconds"].to_string())

    print("\nAll belt events")
    print(sol["belt_sequence"].to_string(index=False))

    belt_df = sol["belt_sequence"].copy()
    belt_df["result_type"] = "belt_event"

    tote_blocks_df = sol["tote_blocks"].reset_index().copy()
    tote_blocks_df["result_type"] = "tote_block"

    tote_item_df = sol["tote_item_sequence"].copy()
    tote_item_df["result_type"] = "tote_item_sequence"

    order_timing_df = pd.DataFrame({
        "order": sol["order_start_seconds"].index,
        "order_start_slot": sol["order_start_slots"].values,
        "order_start_seconds": sol["order_start_seconds"].values,
        "order_completion_slot": sol["order_completion_slots"].values,
        "order_completion_seconds": sol["order_completion_seconds"].values,
    })
    order_pos = {o: i + 1 for i, o in enumerate(sol["order_sequence"])}
    order_timing_df["order_sequence_pos"] = order_timing_df["order"].map(order_pos)
    order_timing_df["result_type"] = "order_timing"

    order_conv_df = pd.DataFrame(
        [{"order": int(o), "conveyor": int(c)} for o, c in sol["order_to_conveyor"].items()]
    )
    order_conv_df["result_type"] = "order_to_conveyor"

    tote_seq_df = pd.DataFrame(
        {"tote": sol["tote_sequence"], "tote_sequence_pos": list(range(1, len(sol["tote_sequence"]) + 1))}
    )
    tote_seq_df["result_type"] = "tote_sequence"

    summary_df = pd.DataFrame(
        [
            {
                "objective_value_seconds": float(sol["objective_value_seconds"]),
                "solver_status": int(sol["solver_status"]),
                "solver_message": str(sol["solver_message"]),
            }
        ]
    )
    summary_df["result_type"] = "summary"

    all_results = pd.concat(
        [
            summary_df,
            order_conv_df,
            tote_seq_df,
            order_timing_df,
            tote_blocks_df,
            tote_item_df,
            belt_df,
        ],
        ignore_index=True,
        sort=False,
    )

    output_file = "MSE433_M3_all_results.csv"
    all_results.to_csv(output_file, index=False)
    print("\nSaved:", output_file)






Solver status
1 Time limit reached. (HiGHS Status 13: Time limit reached)

Order -> conveyor
{1: 0, 2: 0, 3: 0, 4: 1, 5: 0, 6: 0, 7: 0, 8: 0, 9: 1, 10: 0, 11: 0}

Objective (sum completion time in seconds, recirculation-aware)
895.0

Chosen order sequence
[3, 5, 11, 6, 10, 4, 8, 7, 9, 2, 1]

Tote sequence
[7, 11, 12, 10, 1, 9, 8, 3, 13, 14, 4, 0]

Order completion seconds
order
1     180.0
2     155.0
3      10.0
4      75.0
5      20.0
6      40.0
7     115.0
8      85.0
9     135.0
10     55.0
11     25.0

All belt events
 slot  release_time_seconds  pick_time_seconds  unit  order  item_type  tote  conveyor  recirc_loops
    1                   0.0                5.0    10      3          5     7         0             0
    2                   5.0               10.0    11      3          5     7         0             0
    3                  10.0               15.0    16      5          1    11         0             0
    4                  15.0               20.0    17      5      

# 4) Build Picking-System Input File
This block generates MSE433_M3_generated_input.csv in the required conveyor-input format.


In [43]:
# Build picking-system input file with strict order sequencing.
# Rules enforced here:
# 1) complete each order before starting the next order,
# 2) each order is assigned to exactly one conveyor.

import pandas as pd

item_cols = ["cirle", "pentagon", "trapezoid", "triangle", "star", "moon", "heart", "cross"]
MAX_UNITS_PER_ROW = 3  # smaller => more rows per order

itemtypes = pd.read_csv("order_itemtypes.csv", header=None)
quantities = pd.read_csv("order_quantities.csv", header=None)
totes = pd.read_csv("orders_totes.csv", header=None)

rows = []
for o in range(itemtypes.shape[0]):
    for k in range(itemtypes.shape[1]):
        it = itemtypes.iat[o, k]
        qt = quantities.iat[o, k] if k < quantities.shape[1] else pd.NA
        tt = totes.iat[o, k] if k < totes.shape[1] else pd.NA
        if pd.notna(it) and pd.notna(qt) and pd.notna(tt):
            rows.append({"order": o + 1, "item_type": int(it), "quantity": int(qt)})

req = pd.DataFrame(rows)
if req.empty:
    raise ValueError("No valid rows found from CSV files.")
if req["item_type"].min() < 0 or req["item_type"].max() > 7:
    raise ValueError("Item types must be in [0, 7] for this input format.")

order_item = req.groupby(["order", "item_type"], as_index=False)["quantity"].sum()
orders = sorted(order_item["order"].unique().tolist())

# One conveyor per order (deterministic assignment).
# If orders > 4, conveyors are reused for later orders, still one conveyor per order.
order_to_conv = {o: i % 4 for i, o in enumerate(orders)}

schedule_rows = []
for o in orders:
    conv = order_to_conv[o]
    rem = {it: 0 for it in range(8)}
    grp = order_item[order_item["order"] == o]
    for r in grp.itertuples(index=False):
        rem[int(r.item_type)] = int(r.quantity)

    while sum(rem.values()) > 0:
        row = {"conv_num": conv}
        for it in range(8):
            row[it] = 0

        cap = MAX_UNITS_PER_ROW
        item_order = sorted(range(8), key=lambda i: (-rem[i], i))
        for it in item_order:
            if cap == 0:
                break
            take = min(cap, rem[it])
            if take > 0:
                row[it] = take
                rem[it] -= take
                cap -= take

        if sum(row[i] for i in range(8)) == 0:
            raise RuntimeError(f"Could not make progress building rows for order {o}.")

        schedule_rows.append(row)

out = pd.DataFrame(schedule_rows)
out = out[["conv_num"] + list(range(8))]
out.columns = ["conv_num"] + item_cols

output_file = "MSE433_M3_generated_input.csv"
out.to_csv(output_file, index=False)

print("Generated:", output_file)
print(out.to_string(index=False))
print("\nOrder -> conveyor assignment:")
for o in orders:
    print(f"Order {o} -> Conveyor {order_to_conv[o]}")


Generated: MSE433_M3_generated_input.csv
 conv_num  cirle  pentagon  trapezoid  triangle  star  moon  heart  cross
        0      0         0          0         3     0     0      0      0
        0      0         2          0         0     0     0      0      0
        1      0         0          0         3     0     0      0      0
        1      0         0          1         0     1     0      0      0
        2      0         0          0         0     0     2      0      0
        3      3         0          0         0     0     0      0      0
        3      0         0          0         0     0     1      0      0
        0      0         1          1         0     0     0      0      0
        1      0         2          0         0     0     0      0      0
        2      0         0          0         2     0     1      0      0
        3      0         0          1         0     2     0      0      0
        0      3         0          0         0     0     0      0     